# Bagian 5 | Eksperimen LSTM 


In [1]:
import os
import sys
import json
import time
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import pandas as pd
import seaborn as sns
from nltk.translate.bleu_score import corpus_bleu

PROJECT_ROOT = os.path.abspath(os.path.join(os.path.abspath(''), '..', '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.lstm.lstm_keras import LSTMKeras
from src.lstm.lstm_scratch import LSTMScratch

metadata_path = os.path.join(PROJECT_ROOT, 'outputs', 'vocab', 'metadata.json')
vocab_path    = os.path.join(PROJECT_ROOT, 'outputs', 'vocab', 'vocab.json')
test_txt      = os.path.join(PROJECT_ROOT, 'data', 'Flickr_8k.testImages.txt')
images_dir    = os.path.join(PROJECT_ROOT, 'data', 'Images')
captions_file = os.path.join(PROJECT_ROOT, 'data', 'captions.txt')
models_dir    = os.path.join(PROJECT_ROOT, 'models', 'lstm')
output_dir    = os.path.join(PROJECT_ROOT, 'outputs')

with open(vocab_path, 'r') as f:
    word_to_idx = json.load(f)
idx_to_word = {str(idx): word for word, idx in word_to_idx.items()}

with open(test_txt, 'r') as f:
    test_images = [line.strip() for line in f if line.strip()]
sample_images = [img for img in test_images if os.path.exists(os.path.join(images_dir, img))]

def load_captions(captions_file):
    image_captions = {}
    with open(captions_file, 'r', encoding='utf-8') as f:
        next(f)
        for line in f:
            line = line.strip()
            if not line: continue
            parts = line.split(',', 1)
            if len(parts) == 2:
                img, cap = parts
                if img not in image_captions:
                    image_captions[img] = []
                image_captions[img].append(cap)
    return image_captions

image_captions = load_captions(captions_file)
print(f'Test images  : {len(sample_images)}')
print(f'Project root : {PROJECT_ROOT}')


I0000 00:00:1778926312.784068   19008 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Test images  : 1000
Project root : /home/hanifu/CNN_RNN_LSTM


## 1. Perbandingan Keras vs Scratch
Membandingkan prediksi yang dihasilkan oleh model **Keras Native** dan implementasi **From Scratch**.


In [4]:
model_name = 'lstm_L1_H512'
model_path = os.path.join(models_dir, f'{model_name}.keras')

try:
    keras_model   = LSTMKeras(model_path, metadata_path)
    scratch_model = LSTMScratch(model_path, metadata_path)
    print('Berhasil memuat model Keras dan Scratch!')
except Exception as e:
    print(f'Gagal memuat model. Pastikan model {model_name} sudah ada.')
    print('Error:', e)


Encoder: INCEPTIONV3 — output dim: 2048
Gagal memuat model. Pastikan model lstm_L1_H512 sudah ada.
Error: Requested the deserialization of a `Lambda` layer whose `function` is a Python lambda. This carries a potential risk of arbitrary code execution and thus it is disallowed by default. If you trust the source of the artifact, you can override this error by passing `safe_mode=False` to the loading function, or calling `keras.config.enable_unsafe_deserialization().


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
random.seed(42)

for ax in axes:
    img_name = random.choice(sample_images)
    img_path = os.path.join(images_dir, img_name)

    cap_keras   = keras_model.generate_caption(img_path, idx_to_word)
    cap_scratch = scratch_model.generate_caption(img_path, idx_to_word)

    img_data = mpimg.imread(img_path)
    ax.imshow(img_data)
    ax.axis('off')
    ax.set_title(f'Keras  : {cap_keras}\nScratch: {cap_scratch}', fontsize=9, wrap=True)

plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'visualizations', 'lstm_keras_vs_scratch_samples.png'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Perbandingan BLEU-4 dan waktu eksekusi Keras vs Scratch
eval_images = sample_images  

references = []
for img in eval_images:
    img_refs = [cap.lower().replace('.', '').replace(',', '').split()
                for cap in image_captions[img]]
    references.append(img_refs)

keras_vs_scratch_results = []

for name, model in [('Keras', keras_model), ('Scratch', scratch_model)]:
    hypotheses = []
    start_time = time.time()

    for img in eval_images:
        img_path = os.path.join(images_dir, img)
        caption  = model.generate_caption(img_path, idx_to_word)
        words    = [w for w in caption.split() if w not in ['<start>', '<end>', '<pad>']]
        hypotheses.append(words)

    total_time = time.time() - start_time
    avg_time   = total_time / len(eval_images)
    bleu4      = corpus_bleu(references, hypotheses)

    keras_vs_scratch_results.append({'model': name, 'bleu4': bleu4, 'avg_time_s': avg_time})
    print(f'[{name}] BLEU-4: {bleu4:.4f} | Waktu/img: {avg_time:.4f}s')

out_json = os.path.join(output_dir, 'experiment_keras_vs_scratch_lstm.json')
with open(out_json, 'w') as f:
    json.dump(keras_vs_scratch_results, f, indent=4)
print(f'Tersimpan di {out_json}')


## 2. Eksperimen 6 Variasi Arsitektur
Evaluasi BLEU-4 dan waktu inferensi untuk 6 variasi hyperparameter (Layer × Hidden Units).


In [ ]:
def evaluate_models(num_samples=None):
    eval_images = sample_images if num_samples is None else sample_images[:num_samples]

    references = []
    for img in eval_images:
        img_refs = [cap.lower().replace('.', '').replace(',', '').split()
                    for cap in image_captions[img]]
        references.append(img_refs)

    configs = [
        'lstm_L1_H128', 'lstm_L1_H512',
        'lstm_L2_H128', 'lstm_L2_H512',
        'lstm_L3_H128', 'lstm_L3_H512',
    ]

    results = []

    for config in configs:
        model_path = os.path.join(models_dir, f'{config}.keras')
        if not os.path.exists(model_path):
            print(f'Model {config} tidak ditemukan. Skip.')
            continue

        print(f'Mengevaluasi {config}...')
        captioner = LSTMScratch(model_path, metadata_path)

        hypotheses = []
        start_time = time.time()

        for img in eval_images:
            img_path = os.path.join(images_dir, img)
            caption  = captioner.generate_caption(img_path, idx_to_word)
            words    = [w for w in caption.split() if w not in ['<start>', '<end>', '<pad>']]
            hypotheses.append(words)

        total_time = time.time() - start_time
        avg_time   = total_time / len(eval_images)
        bleu4      = corpus_bleu(references, hypotheses)

        results.append({
            'config': config,
            'bleu4': bleu4,
            'avg_inference_time_s': avg_time,
        })

    print('\nHasil Evaluasi:')
    for r in results:
        print(f"{r['config']:<15} | BLEU-4: {r['bleu4']:.4f} | Waktu/img: {r['avg_inference_time_s']:.4f}s")

    out_json = os.path.join(output_dir, 'experiment_6_lstm_variations.json')
    with open(out_json, 'w') as f:
        json.dump(results, f, indent=4)
    print(f'Tersimpan di {out_json}')
    return results

lstm_results = evaluate_models(num_samples=len(sample_images))


## 3. Visualisasi Hasil Eksperimen


In [ ]:
res_path = os.path.join(output_dir, 'experiment_6_lstm_variations.json')
if os.path.exists(res_path):
    with open(res_path, 'r') as f:
        data = json.load(f)

    df = pd.DataFrame(data)

    fig, ax1 = plt.subplots(figsize=(10, 5))
    sns.barplot(data=df, x='config', y='bleu4', color='steelblue', ax=ax1, label='BLEU-4')
    ax1.set_ylabel('BLEU-4 Score')
    ax1.tick_params(axis='x', rotation=45)

    ax2 = ax1.twinx()
    sns.lineplot(data=df, x='config', y='avg_inference_time_s', color='red', marker='o', ax=ax2, label='Inference Time (s)')
    ax2.set_ylabel('Inference Time (seconds)')

    plt.title('BLEU-4 Score and Inference Time across LSTM Variations')
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'visualizations', 'lstm_bleu_inference.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Belum ada data evaluasi. Jalankan sel sebelumnya dulu.')


## 4. Evaluasi Pengaruh Maksimum Panjang Caption


In [ ]:
def evaluate_max_lengths(num_samples=100):
    eval_images = sample_images[:num_samples]

    references = []
    for img in eval_images:
        img_refs = [cap.lower().replace('.', '').replace(',', '').split()
                    for cap in image_captions[img]]
        references.append(img_refs)

    max_lengths = [10, 20, 30]
    results = []

    for max_len in max_lengths:
        hypotheses = []
        print(f'Mengevaluasi max_length={max_len}...')

        for img in eval_images:
            img_path = os.path.join(images_dir, img)
            caption  = keras_model.generate_caption(img_path, idx_to_word, max_len=max_len)
            words    = [w for w in caption.split() if w not in ['<start>', '<end>', '<pad>']]
            hypotheses.append(words)

        bleu4 = corpus_bleu(references, hypotheses)
        results.append({'max_len': max_len, 'bleu4': bleu4})

    print('\nHasil Evaluasi Max Length:')
    for r in results:
        print(f"Max Len: {r['max_len']} | BLEU-4: {r['bleu4']:.4f}")

    out_json = os.path.join(output_dir, 'experiment_max_len_lstm.json')
    with open(out_json, 'w') as f:
        json.dump(results, f, indent=4)
    print(f'Tersimpan di {out_json}')

evaluate_max_lengths(num_samples=100)
